# 05 Materials Project Screening

This notebook inspects the screening table, demonstrates the filtering logic for rescued candidates, and documents the optional CIF acquisition workflow.

In [1]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(repo_root)


/root/大创/perovskite-bandgap-ml-correction


In [2]:
import os
import pandas as pd

screen = pd.read_csv(repo_root / 'data' / 'mp_screening_results.csv')
print(screen.shape)
screen.head()


(6380, 287)


,frac f valence electrons,cohesive energy (MP),MagpieData range GSbandgap,MagpieData mode Row,MagpieData mode MeltingT,F fraction,Al fraction,MagpieData avg_dev Row,MagpieData mode NfUnfilled,Pm fraction,...,MagpieData minimum NpUnfilled,K fraction,MagpieData mode GSbandgap,MagpieData avg_dev NfUnfilled,MagpieData avg_dev Electronegativity,MagpieData avg_dev Column,is_metal_gga,Predicted_Is_Metal,Predicted_Raw_Gap,Final_Predicted_Bandgap
0,0.175000,1.806262,7.853,5,386.85,0.0,0.0,1.36,0,0.0,...,0,0.0,1.062,0.00,0.1896,4.88,1,1,0.873283,0.000000
1,0.222222,5.003612,1.524,5,1828.05,0.0,0.0,1.04,0,0.0,...,0,0.0,0.000,1.28,0.3008,2.48,1,1,1.727649,0.000000
2,0.200000,5.871092,0.000,2,54.80,0.0,0.0,1.44,0,0.0,...,0,0.0,0.000,2.24,0.9276,5.04,1,0,2.377517,2.377517
3,0.368421,6.747693,0.000,2,54.80,0.0,0.0,1.20,0,0.0,...,0,0.0,0.000,0.00,1.0248,6.24,0,0,1.510636,1.510636
4,0.000000,6.451641,0.000,2,54.80,0.0,0.0,0.96,0,0.0,...,0,0.0,0.000,0.00,1.0200,6.00,1,1,2.032451,0.000000


In [3]:
required_cols = ['pretty_formula', 'band_gap', 'Predicted_Is_Metal', 'Final_Predicted_Bandgap', 'formation_energy_per_atom']
missing = [c for c in required_cols if c not in screen.columns]
print('missing columns:', missing)


missing columns: []


In [4]:
rescued = screen[(screen['is_metal_gga'] == 1) & (screen['Predicted_Is_Metal'] == 0)]
pb_free = rescued[~rescued['pretty_formula'].str.contains('Pb', na=False)]
stable = pb_free[pb_free['formation_energy_per_atom'] < 0].copy()
stable['sq_diff'] = (stable['Final_Predicted_Bandgap'] - 1.34).abs()
top_candidates = stable.sort_values('sq_diff')[['pretty_formula', 'band_gap', 'Final_Predicted_Bandgap', 'formation_energy_per_atom']].head(10)
print({'rescued': len(rescued), 'pb_free': len(pb_free), 'stable': len(stable)})
top_candidates


{'rescued': 462, 'pb_free': 457, 'stable': 452}


,pretty_formula,band_gap,Final_Predicted_Bandgap,formation_energy_per_atom
4644,Lu2TiCuO6,0.0,1.340393,-3.175632
5088,Na2TlCuCl6,0.0,1.336851,-1.326403
5874,Cs2TlHgI6,0.0,1.350929,-0.814937
4827,K2GaHgBr6,0.0,1.328317,-1.026372
5570,Na2AlHgCl6,0.0,1.359102,-1.557958
5815,K2InAgI6,0.0,1.362654,-0.774312
5827,K2TlAuI6,0.0,1.315341,-0.672502
1372,MgZnO3,0.0,1.364933,-1.241889
4749,Ba2CuTeO6,0.0,1.311406,-2.165120
3750,K2NaTaI6,0.0,1.371204,-0.860437


In [5]:
print('MP_API_KEY set:', bool(os.getenv('MP_API_KEY')))
print('available CIF files:', len(list((repo_root / 'data' / 'cifs').glob('*.cif'))))


MP_API_KEY set: False
available CIF files: 0


In [6]:
from src.screening import fetch_cifs
print('Run fetch_cifs() only after setting MP_API_KEY in the environment.')


Run fetch_cifs() only after setting MP_API_KEY in the environment.
